In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("GALAXY_DATASET/final_15000.csv")
df.head()

In [ ]:
# 10 images are missing from the folder, drop those rows
img_folder = "Images Differing sizes/224"
valid = set(int(f.replace(".jpg", "")) for f in os.listdir(img_folder) if f.endswith(".jpg"))
df = df[df["asset_id"].isin(valid)].reset_index(drop=True)
print(len(df))  # should be 14990

In [ ]:
def make_label(row, thresh=0.5):
    smooth = row["t01_smooth_or_features_a01_smooth_debiased"]
    featured = row["t01_smooth_or_features_a02_features_or_disk_debiased"]
    edgeon = row["t02_edgeon_a04_yes_debiased"]
    bar = row["t03_bar_a06_bar_debiased"]
    spiral = row["t04_spiral_a08_spiral_debiased"]

    if smooth >= thresh:
        return "elliptical"
    elif featured >= thresh:
        if edgeon >= thresh:
            return "edge_on"
        elif bar >= thresh and spiral >= thresh:
            return "barred_spiral"
        elif spiral >= thresh:
            return "spiral"
    return None

df["label"] = df.apply(make_label, axis=1)
df = df.dropna(subset=["label"]).reset_index(drop=True)
print(df["label"].value_counts())

In [ ]:
classes = ["elliptical", "spiral", "barred_spiral", "edge_on"]
label2idx = {c: i for i, c in enumerate(classes)}
df["label_idx"] = df["label"].map(label2idx)
label2idx